<a href="https://colab.research.google.com/github/Yuneak/NLP/blob/main/NLP_Assignemnt_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install kaggle pandas

In [ ]:
import kaggle

In [ ]:
!kaggle datasets files nolanbconaway/24169-pitchfork-reviews

name               size  creationDate                
------------  ---------  --------------------------  
data.sqlite3  157839360  2024-11-17 21:15:17.420000  


In [ ]:
!kaggle datasets download -d nolanbconaway/24169-pitchfork-reviews

Dataset URL: https://www.kaggle.com/datasets/nolanbconaway/24169-pitchfork-reviews
License(s): unknown
24169-pitchfork-reviews.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
import zipfile
import os

zip_file = "24169-pitchfork-reviews.zip"

if os.path.exists(zip_file):
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Extraction complete!")
else:
    print("Zip file not found.")

Extraction complete!


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('data.sqlite3')
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("Tables found in database:", tables)

Tables found in database: [('artists',), ('reviews',), ('artist_review_map',), ('author_review_map',), ('genre_review_map',), ('tombstones',), ('tombstone_label_map',), ('tombstone_release_year_map',)]


In [ ]:
df_peek = pd.read_sql_query("select * from reviews limit 5", conn)

print("Columns in 'reviews' table:", df_peek.columns.tolist())
df_peek.head()

Columns in 'reviews' table: ['review_url', 'is_standard_review', 'pub_date', 'body']


,review_url,is_standard_review,pub_date,body
0,/reviews/albums/9232-vesikansi/,1,2006-07-27 06:00:04,"Over the past few years, several of Finland's ..."
1,/reviews/albums/22823-elsewhere/,1,2017-01-27 06:00:00,It's hard not to root for Pinegrove. Even befo...
2,/reviews/albums/4703-le-tigre/,1,1999-10-26 05:00:03,Kathleen Hanna is best known as the former lea...
3,/reviews/albums/21978-stranger-to-stranger/,1,2016-06-09 05:00:00,Of all the baby-boomer heroes to make it past ...
4,/reviews/albums/11996-everyone-is-crying-out-t...,1,2008-07-23 06:00:05,Yanka Dyagileva was a Russian poet and punk/fo...


In [ ]:
df_tombstones = pd.read_sql_query("SELECT * FROM tombstones LIMIT 5", conn)

print("Columns in 'tombstones' table:", df_tombstones.columns.tolist())
df_tombstones.head()

Columns in 'tombstones' table: ['review_tombstone_id', 'review_url', 'picker_index', 'title', 'score', 'best_new_music', 'best_new_reissue']


,review_tombstone_id,review_url,picker_index,title,score,best_new_music,best_new_reissue
0,/reviews/albums/9232-vesikansi/-0,/reviews/albums/9232-vesikansi/,0,Vesikansi,7.1,0.0,NaN
1,/reviews/albums/22823-elsewhere/-0,/reviews/albums/22823-elsewhere/,0,Elsewhere,7.5,0.0,0.0
2,/reviews/albums/4703-le-tigre/-0,/reviews/albums/4703-le-tigre/,0,Le Tigre,8.5,NaN,NaN
3,/reviews/albums/21978-stranger-to-stranger/-0,/reviews/albums/21978-stranger-to-stranger/,0,Stranger to Stranger,7.2,0.0,0.0
4,/reviews/albums/11996-everyone-is-crying-out-t...,/reviews/albums/11996-everyone-is-crying-out-t...,0,"Everyone Is Crying Out to Me, Beware",7.4,0.0,NaN


In [ ]:
query = """
select tombstones.score,
reviews.body
from tombstones
join reviews on tombstones.review_url = reviews.review_url
"""

df = pd.read_sql_query(query, conn)

df = df.rename(columns={'body': 'text'})

print(f"Successfully combined {len(df)} reviews with their scores.")
df.head

Successfully combined 25212 reviews with their scores.


<bound method NDFrame.head of        score                                               text
0        7.1  Over the past few years, several of Finland's ...
1        7.5  It's hard not to root for Pinegrove. Even befo...
2        8.5  Kathleen Hanna is best known as the former lea...
3        7.2  Of all the baby-boomer heroes to make it past ...
4        7.4  Yanka Dyagileva was a Russian poet and punk/fo...
...      ...                                                ...
25207    7.6  In his book Pimp: The Story of My Life, author...
25208    7.6  Angry Angles captured Jay Reatard at a pivotal...
25209    7.2  Male-female duos are nothing new in rock, but ...
25210    7.4  Just as Evan Dando's long relationship with su...
25211    7.8  New indie development: Tweemo wimps are gettin...

[25212 rows x 2 columns]>

In [ ]:
def assign_sentiment(score):
    if score < 5.0:
        return 0 #Negative
    elif score < 7.0:
        return 1 #Neutral
    else:
        return 2 #Positive

df['label'] = df['score'].apply(assign_sentiment)

print("Count of reviews per tier:")
print(df['label'].value_counts().sort_index())

print("\nPercentage breakdown:")
print(df['label'].value_counts(normalize=True).sort_index() * 100)

Count of reviews per tier:
label
0     1445
1     7591
2    16176
Name: count, dtype: int64

Percentage breakdown:
label
0     5.731398
1    30.108678
2    64.159924
Name: proportion, dtype: float64


In [ ]:
!kaggle datasets files kauvinlucas/30000-albums-aggregated-review-ratings

name                                   size  creationDate                
---------------------------------  --------  --------------------------  
Review excerpts for NLP/test.csv   10836016  2021-04-13 01:06:51.726000  
Review excerpts for NLP/train.csv  25288983  2021-04-13 01:06:52.460000  
album_ratings.csv                   2741344  2021-04-13 01:06:52.002000  


In [ ]:
!kaggle datasets download -d kauvinlucas/30000-albums-aggregated-review-ratings

Dataset URL: https://www.kaggle.com/datasets/kauvinlucas/30000-albums-aggregated-review-ratings
License(s): GPL-2.0
30000-albums-aggregated-review-ratings.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
with zipfile.ZipFile("30000-albums-aggregated-review-ratings.zip", 'r') as zip_ref:
    zip_ref.extractall("contemporary_data")

print("Download and extraction complete.")

Download and extraction complete.


In [ ]:
print(os.listdir("contemporary_data"))

['Review excerpts for NLP', 'album_ratings.csv']


In [ ]:
file_path = os.path.join("contemporary_data", "album_ratings.csv")

df_test_inspect = pd.read_csv(file_path, nrows=5)

print("Columns found in test.csv:")
print(df_test_inspect.columns.tolist())

# Display the first 5 records
df_test_inspect.head()

Columns found in test.csv:
['Artist', 'Title', 'Release Month', 'Release Day', 'Release Year', 'Format', 'Label', 'Genre', 'Metacritic Critic Score', 'Metacritic Reviews', 'Metacritic User Score', 'Metacritic User Reviews', 'AOTY Critic Score', 'AOTY Critic Reviews', 'AOTY User Score', 'AOTY User Reviews']


,Artist,Title,Release Month,Release Day,Release Year,Format,Label,Genre,Metacritic Critic Score,Metacritic Reviews,Metacritic User Score,Metacritic User Reviews,AOTY Critic Score,AOTY Critic Reviews,AOTY User Score,AOTY User Reviews
0,Neko Case,Middle Cyclone,March,3,2009,LP,ANTI-,Alt-Country,79,31,8.7,31,79,25,78,55
1,Jason Isbell & The 400 Unit,Jason Isbell & The 400 Unit,February,17,2009,LP,Thirty Tigers,Country Rock,70,14,8.4,7,73,11,73,8
2,Animal Collective,Merriweather Post Pavilion,January,20,2009,LP,Domino,Psychedelic Pop,89,36,8.5,619,92,30,87,1335
3,Bruce Springsteen,Working on a Dream,January,27,2009,LP,Columbia Records,Rock,72,29,7.9,101,70,23,66,38
4,Andrew Bird,Noble Beast,January,20,2009,LP,Fat Possum,Singer-Songwriter,79,29,8.7,47,74,24,78,44


In [ ]:
print(os.listdir("contemporary_data/Review excerpts for NLP"))

['test.csv', 'train.csv']


In [ ]:
aug_file_path = os.path.join(
    "contemporary_data",
    "Review excerpts for NLP",
    "test.csv"   # <-- actual file name here
)

aug_df = pd.read_csv(aug_file_path, nrows=5)
aug_df.head()

,Artist,Title,Source,Rating,Review,Reception
0,The Bird and the Bee,The Bird and the Bee,Delusions of Adequacy,70,"The Bird and the Bee is as light as a feather,...",Favorable
1,Pet Shop Boys,Yes,The A.V. Club,67,"On the surface, this big-sounding album belong...",Favorable
2,The Rapture,In The Grace of Your Love,Clash Music,70,"It's not a difficult or aloof album, but there...",Favorable
3,Death Cab For Cutie,Codes and Keys,Boston Globe,70,Here on Death Cab for Cutie's seventh record t...,Favorable
4,French Kicks,Two Thousand,Filter,78,[They've] caught on to what all those indies d...,Favorable


In [ ]:
print("Columns in the Augmentation Dataset:")
print(aug_df.columns.tolist())

# Let's also see the first row to confirm where the text is
print("\nFirst review excerpt sample:")
print(aug_df.iloc[0])

Columns in the Augmentation Dataset:
['Artist', 'Title', 'Source', 'Rating', 'Review', 'Reception']

First review excerpt sample:
Artist                                    The Bird and the Bee
Title                                     The Bird and the Bee
Source                                   Delusions of Adequacy
Rating                                                      70
Review       The Bird and the Bee is as light as a feather,...
Reception                                            Favorable
Name: 0, dtype: object


In [ ]:
aug_df_full = pd.read_csv(aug_file_path)

# 2. Define the 100-point scale mapping
def map_100_scale(score):
    if score < 50:
        return 0  # Negative
    elif score < 60:
        return 1  # Neutral
    else:
        return 2  # Positive

# 3. Apply the mapping
aug_df_full['label'] = aug_df_full['Rating'].apply(map_100_scale)

# 4. Filter for ONLY Negative and Neutral to bolster our minority classes
df_extra_neg = aug_df_full[aug_df_full['label'] == 0][['Review', 'label']]
df_extra_neu = aug_df_full[aug_df_full['label'] == 1][['Review', 'label']]

# Rename columns to match your Pitchfork DataFrame
df_extra_neg.columns = ['text', 'label']
df_extra_neu.columns = ['text', 'label']

print(f"Extracted {len(df_extra_neg)} new Negative reviews.")
print(f"Extracted {len(df_extra_neu)} new Neutral reviews.")

Extracted 2933 new Negative reviews.
Extracted 2662 new Neutral reviews.


In [ ]:
df_combined = pd.concat([df[['text', 'label']], df_extra_neg, df_extra_neu], ignore_index=True)

print("--- NEW DISTRIBUTION ---")
print(df_combined['label'].value_counts())

--- NEW DISTRIBUTION ---
label
2    16176
1    10253
0     4378
Name: count, dtype: int64


In [ ]:
aug_file_path2 = os.path.join(
    "contemporary_data",
    "Review excerpts for NLP",
    "train.csv"   # <-- actual file name here
)

aug_df2 = pd.read_csv(aug_file_path2)
aug_df2.head()

,Artist,Title,Source,Rating,Review,Reception
0,Jason Isbell & The 400 Unit,Jason Isbell & The 400 Unit,Spin,80,"With Drive-By Truckers, singer-guitarist Jason...",Favorable
1,Jason Isbell & The 400 Unit,Jason Isbell & The 400 Unit,AllMusic,80,"Just barely out of his twenties, he writes wit...",Favorable
2,Jason Isbell & The 400 Unit,Jason Isbell & The 400 Unit,Uncut,80,This is best thought of as ‘country soul’. Isb...,Favorable
3,Jason Isbell & The 400 Unit,Jason Isbell & The 400 Unit,Pitchfork,74,Many of the songs on Isbell's sophomore releas...,Favorable
4,Jason Isbell & The 400 Unit,Jason Isbell & The 400 Unit,Austin Chronicle,67,The horns and soul on 'No Choice in the Matter...,Favorable


In [ ]:
aug_df2['label'] = aug_df2['Rating'].apply(map_100_scale)

df_extra_neg2 = aug_df2[aug_df2['label'] == 0][['Review', 'label']]
df_extra_neu2 = aug_df2[aug_df2['label'] == 1][['Review', 'label']]

# Rename columns to match your Pitchfork DataFrame
df_extra_neg2.columns = ['text', 'label']
df_extra_neu2.columns = ['text', 'label']

print(f"Extracted {len(df_extra_neg2)} new Negative reviews.")
print(f"Extracted {len(df_extra_neu2)} new Neutral reviews.")

Extracted 6748 new Negative reviews.
Extracted 6371 new Neutral reviews.


In [ ]:
df_combined2 = pd.concat([df[['text', 'label']], df_extra_neg2, df_extra_neu2], ignore_index=True)

print("--- NEW DISTRIBUTION ---")
print(df_combined2['label'].value_counts())

--- NEW DISTRIBUTION ---
label
2    16176
1    13962
0     8193
Name: count, dtype: int64


In [ ]:
!pip install ftfy

In [ ]:
import ftfy
import re
import string

def clean_review_text(text):
    if not isinstance(text, str):
        return ""

    # 1. FIX ENCODING (The most important step for those â€ artifacts)
    text = ftfy.fix_text(text)

    # 1. Lowercase everything (NNLM is case-sensitive, so we standardize)
    text = text.lower()

    # 2. Remove HTML tags like <br> or <div> (common in Kaggle/Scraped data)
    text = re.sub(r'<.*?>', '', text)

    # 3. Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # 4. Remove Newlines and Tabs (\n, \t)
    text = re.sub(r'[\n\t\r]', ' ', text)

    # 5. Remove Punctuation
    # (Note: We keep it simple to avoid losing 'sentiment' context)
    text = text.translate(str.maketrans('', '', string.punctuation))

    # 6. Strip extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply the cleaning to your combined dataframe
# Assuming your main dataframe is 'df_combined'
df_combined2['clean_text'] = df_combined2['text'].apply(clean_review_text)

# Let's see a 'Before' and 'After' to verify
print("--- ORIGINAL TEXT (First 150 chars) ---")
print(df_combined2['text'].iloc[0][:150])

print("\n--- CLEANED TEXT (First 150 chars) ---")
print(df_combined2['clean_text'].iloc[0][:150])



--- ORIGINAL TEXT (First 150 chars) ---
Over the past few years, several of Finland's various interrelated underground acts-- Kemialliset Ystävät, ES, Kuupuu, the Anaksimandros-- have engage

--- CLEANED TEXT (First 150 chars) ---
over the past few years several of finlands various interrelated underground acts kemialliset ystävät es kuupuu the anaksimandros have engaged in what


In [ ]:
# Test it on your specific examples
test_cases = ["â€œSorryâ€", "â€œCanâ€™t Say Noâ€"]
for t in test_cases:
    print(f"Original: {t} -> Cleaned: {clean_review_text(t)}")

Original: â€œSorryâ€ -> Cleaned: sorry
Original: â€œCanâ€™t Say Noâ€ -> Cleaned: cant say no


In [ ]:
!pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 10.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 12.6 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]


In [ ]:
from sklearn.model_selection import train_test_split

# X is your cleaned text, y is your labels (0, 1, 2)
X = df_combined2['clean_text'].values
y = df_combined2['label'].values

# Split: 80% Training, 20% Testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training Samples: {len(X_train)}")
print(f"Testing Samples: {len(X_test)}")

Training Samples: 30664
Testing Samples: 7667


In [ ]:
!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.6/572.6 MB 23.4 MB/s  0:00:25m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 20.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 17.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 20.7 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 20.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 23.3 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20/20 [tensorflow]0 [tensorflow]-py]


In [ ]:
!pip install tensorflow_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 11.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0/2 [tf-keras]

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub

# 1. Load the Embedding Layer
# We use 'trainable=True' so the model can learn music-specific context
model_url = "https://tfhub.dev/google/nnlm-en-dim50-with-normalization/2"
hub_layer = hub.KerasLayer(model_url, input_shape=[], dtype=tf.string, trainable=True)

# 2. Define the Sequential Architecture
model_50 = tf.keras.Sequential([
    hub_layer,                          # Input + Embedding
    tf.keras.layers.Dense(16, activation='relu'), # Hidden "Reasoning" Layer
    tf.keras.layers.Dropout(0.2),       # Dropout to prevent overfitting
    tf.keras.layers.Dense(3, activation='softmax') # Output (3 Tiers: 0, 1, 2)
])

# 3. Compile the Model
model_50.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_50.summary()

I0000 00:00:1774959603.905077     520 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1774959605.460665     520 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774959650.242095     520 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


ModuleNotFoundError: No module named 'tensorflow_hub'